# Data understanding


In [ ]:
# importing libraries
import pandas as pd
import sqlite3

In [ ]:
# loading data sets
bom_df= pd.read_csv("data/bom.movie_gross.csv")
rt_reviews_df=pd.read_csv("data/rt.reviews.tsv", sep= '\t', encoding= 'latin1') 
rt_movie_info_df= pd.read_csv("data/rt.movie_info.tsv", sep= '\t', encoding= 'latin1')
tmdb_movies_df= pd.read_csv("data/tmdb.movies.csv")
tn_movie_budgets_df= pd.read_csv("data/tn.movie_budgets.csv")

# previewing each
for name, df in [('Box office mojo', bom_df), ('Rotten tomatoes Reviews', rt_reviews_df), ('Rotten tomatoes movie info', rt_movie_info_df),
                 ('The movie DB',tmdb_movies_df), ('The numbers', tn_movie_budgets_df)]:
    print(f"\n{name} Sample:")
    print(df.head())
    print(df.info())


## Data understanding of each dataframe

In [ ]:
#Box office mojo dataset(movie_gross)
# checking the shape
bom_df= pd.read_csv("data/bom.movie_gross.csv")
bom_df.shape

bom_df dataframe from Box office Mojo has 3387 records and 5 columns.

In [ ]:
# checking the info
bom_df.info()

## Observation

Total rows: 3387 movies

Columns: 5 - title, studio, domestic_gross, foreign_gross and year.

The dataset is mostly complete with title and year having no missing values

The column with more missing values is foreign_gross followed by domestic_gross and lastly studio.

foreign_gross is stored as object instead of float, likely due to formatting issues.


In [ ]:
# reviewing Rotten Tomatoes dataset(reviews df)
rt_reviews_df=pd.read_csv("data/rt.reviews.tsv", sep= '\t', encoding= 'latin1') 
rt_reviews_df.info()

In [ ]:
# checking missing values
rt_reviews_df.isna().sum()

## Observation

Total rows: 54,432 reviews

Columns: 8 including id, review, rating, fresh, critic, top_critic, publisher, and date

All rows have id, fresh, top_critic, and date

missing values are significant in some columns

- review: 5563 missing values

- rating: 13517 missing

- rating: 13517 missing

- publisher: 309 missing values

Date column can be converted to date-time format



In [ ]:
# reviewing Rotten Tomatoes dataset(movie_info)
rt_movie_info_df.info()

In [ ]:
rt_movie_info_df.isna().sum()

## Observation
Total rows: 1560 movie reviews
    
Columns: 12- id, synopsis, rating, genre, director, writer, theater_date, dvd_date, currency, box_office, runtime, studio

Columns like id, rating, genre, and runtime are mostly complete.

Missing values are significant in several columns

- synopsis: 62 missing

- director: 199 missing

- writer: 449 missing

- theater_date and dvd_date: both have 359 missing

- box_office and currency: 1220 missing

- studio: 1066 missing, most data missing

Theater_date, dvd_date, and runtime should be converted to datetime or numeric for analysis.

box_office and currency are object types need to be converted to numeric




In [ ]:
#  reviewing the movie DB dataset(tmdb_movie df)
tmdb_movies_df.info()

## Observation

Total movies: 26,517

Columns: 10 - includes title info, ratings, language, and release_date
    
No missing values across any of the 10 columns - very clean.

unnamed column needs to be dropped

release_date should be converted to datetime

In [ ]:
# reviewing The Numbers dataset(movie_budgets df)
tn_movie_budgets_df.info()

## Observation

Total entries: 5,782 movies

Columns: 6 - focused on financial and release information

Complete data (no missing entries) across all columns

Key financial metrics included:

- production_budget

- domestic_gross

- worldwide_gross

release_date is present, enabling time-based trend analysis

In [ ]:
# loading the sqlite database
con= sqlite3.connect("data/im.db") #creating a connection

# previewing the schema
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type= "table";
""", con)

In [ ]:
# selecting movie_basics and movie_ratings tables
query= """
SELECT *
FROM movie_basics

"""
pd.read_sql(query, con).tail()

In [ ]:
query= """
SELECT *
FROM movie_ratings
"""
pd.read_sql(query, con).tail()

In [ ]:
# joining the two tables using movie_id
query= """
SELECT *
FROM movie_basics
LEFT JOIN movie_ratings
USING(movie_id);
"""
movie_basics_rating_df= pd.read_sql(query, con)
movie_basics_rating_df.tail()

# Data cleaning

## Box office mojo dataset

In [ ]:
# creating a working copy 
bom_clean_df= bom_df.copy()

In [ ]:
bom_clean_df.info()

In [ ]:
# checking for duplicates
duplicated_rows= bom_clean_df.duplicated()
duplicated_rows.sum()

In [ ]:
# cleaning the bom_df(movie_gross) from Box office mojo dataset
bom_clean_df.columns

In [ ]:
# cleaning title column
# checking for duplicates
key_columns= ['title', 'year']
dupl_row= bom_clean_df[bom_clean_df.duplicated(subset= key_columns, keep= False)]
dupl_row

There are no duplicates

In [ ]:
bom_clean_df.isna().sum()

In [ ]:
# cleaning studio column
bom_clean_df['studio'] = bom_clean_df['studio'].str.strip()

In [ ]:
# checking for missing values 
missing_studio = bom_clean_df['studio'].isnull().sum()
missing_studio

# filling in missing values
bom_clean_df['studio']= bom_clean_df['studio'].fillna("unknown")

In [ ]:
# checking for missing values
bom_clean_df['studio'].isnull().sum()

In [ ]:
missing_domestic_gross = bom_clean_df[bom_clean_df['domestic_gross'].isna()]
# Print the rows with missing domestic_gross
print("\nRows with missing domestic_gross:")
print(missing_domestic_gross[['title', 'studio', 'domestic_gross', 'foreign_gross', 'year']])

In [ ]:
# filling in missing values with mean
mean= bom_clean_df['domestic_gross'].mean()
bom_clean_df['domestic_gross']= bom_clean_df['domestic_gross'].fillna(mean)


# checking for null
bom_clean_df['domestic_gross'].isnull().sum()



In [ ]:
# cleaning year column
bom_clean_df["year"].dtype

In [ ]:
# cleaning foreign column
bom_clean_df['foreign_gross'].isnull().sum()


In [ ]:
# dropping foreign_gross column
bom_clean_df.drop(columns= 'foreign_gross', inplace= True)

## Cleaning Rotten Tomatoes reviews dataset


In [ ]:
# creating a copy
rt_reviews_clean_df=rt_reviews_df.copy()

In [ ]:
# missing values
rt_reviews_clean_df.isna().sum()

In [ ]:
# checking for id duplicates
duplicate_count = rt_reviews_clean_df.duplicated().sum()
print(f"Number of duplicates: {duplicate_count}")

In [ ]:
# checking for duplicates while focusing on specific columns for clarity
if all(col in rt_reviews_clean_df for col in ['id', 'review', 'critic','publisher', 'date']):
        initial_rows_rt_reviews = len(rt_reviews_clean_df)
        rt_reviews_clean_df.drop_duplicates(subset=['id', 'review', 'critic','publisher', 'date'])
        print(f"rt_reviews_clean_df: Removed {initial_rows_rt_reviews - len(rt_reviews_clean_df)} duplicates based on 'id', 'review','publisher', 'critic', and 'date'.")
else:
        print("rt_reviews_clean_df: Skipping duplicate check due to missing 'id', 'review', 'critic', or 'date' columns.")

In [ ]:
# checking for columns unique values in each column
for column in rt_reviews_clean_df.columns:
    print(f"\n{column} unique values:")
    print(rt_reviews_df[column].unique())

In [ ]:
# filling in review column
if 'review' in rt_reviews_clean_df.columns:
    missing_review_count= rt_reviews_clean_df['review'].isna().sum()
    if missing_review_count >0:
        rt_reviews_clean_df['review']= rt_reviews_clean_df['review'].fillna('No Review Text')
        print(f"Filled {missing_review_count} missing 'review' values with 'No Review Text'.")

In [ ]:
# dropping unuseful columns
rt_reviews_clean_df.drop(columns=['rating'], inplace=True)

In [ ]:
# checking if the columns are dropped
rt_reviews_clean_df.columns

In [ ]:
# filling critic column missing values with unknown
rt_reviews_clean_df['critic']= rt_reviews_clean_df['critic'].fillna("Unknown")

In [ ]:
# filling publisher missing with unknown
rt_reviews_clean_df['publisher']= rt_reviews_clean_df['publisher'].fillna("Unknown")

In [ ]:
# converting date into datetime format
rt_reviews_clean_df['date'] = pd.to_datetime(rt_reviews_clean_df['date'])

# checking effectiveness
rt_reviews_clean_df["date"].dtype

In [ ]:
# checking missing values in the df
rt_reviews_clean_df.isna().sum()

## Cleaning Rotten Tomatoes movie info dataset

In [ ]:
# creating a working copy
rt_movie_info_clean= rt_movie_info_df.copy()


In [ ]:
# checking for missing values
rt_movie_info_clean.isnull().sum()

In [ ]:
# checking for columns unique values in each column
for column in rt_movie_info_clean.columns:
    print(f"\n{column} unique values:")
    print(rt_movie_info_clean[column].unique())

## Handling missing values column by column


### Filling in categorical columns with 'unknown'


In [ ]:

for col in ['synopsis', 'rating', 'genre', 'director', 'writer', 'studio']:
    if col in rt_movie_info_clean.columns:
            initial_missing_count = rt_movie_info_clean[col].isnull().sum()
            if initial_missing_count > 0:
                rt_movie_info_clean[col] = rt_movie_info_clean[col].fillna('Unknown')
                print(f"Filled {initial_missing_count} missing '{col}' values with 'Unknown'.")


In [ ]:
rt_movie_info_clean.isna().sum()

In [ ]:
# converting theatre date and dvd date into datetime format
rt_movie_info_clean['theater_date'] = pd.to_datetime(rt_movie_info_clean['theater_date'], errors='coerce')
rt_movie_info_clean['dvd_date'] = pd.to_datetime(rt_movie_info_clean['dvd_date'], errors='coerce')

In [ ]:
# filling in missing runtime values with median runtime
rt_movie_info_clean["runtime"]= rt_movie_info_clean["runtime"].astype(str).str.replace('minutes', '', regex= False)
rt_movie_info_clean["runtime"]= pd.to_numeric(rt_movie_info_clean["runtime"], errors= 'coerce')
# filling missing values with median values
runtime_median= rt_movie_info_clean["runtime"].median()                                     
rt_movie_info_clean["runtime"]= rt_movie_info_clean["runtime"].fillna(runtime_median)

In [ ]:
rt_movie_info_clean.isna().sum()

In [ ]:
# dropping missing theater date and dvd date missing values
rt_movie_info_clean = rt_movie_info_clean.drop(columns=['currency', 'box_office'])


## Cleaning tmdb_movies_df(The movie DB dataset)

In [ ]:
tmdb_movies_df.info()

In [ ]:
# dropping unnamed column
tmdb_movies_df.drop(columns=["Unnamed: 0", 'genre_ids'], inplace=True)

In [ ]:
# checking for unique values in each column
for column in tmdb_movies_df.columns:
    print(f"\n{column} unique values:")
    print(tmdb_movies_df[column].unique())

In [ ]:
# standardizing categorical columns for consistency (lowercase and strip)
tmdb_movies_df['original_language']= tmdb_movies_df['original_language'].str.strip().str.lower()
tmdb_movies_df['original_title']= tmdb_movies_df['original_title'].str.strip()
tmdb_movies_df['title']= tmdb_movies_df['title'].str.strip()

In [ ]:
# converting release date to datetime format
tmdb_movies_df["release_date"] = pd.to_datetime(tmdb_movies_df["release_date"], errors='coerce')

In [ ]:
# extract year and month from movie_budgets tmdb df
# tmdb_movies_df['release_year'] = tmdb_movies_df['release_date'].dt.year
# tmdb_movies_df['release_month'] = tmdb_movies_df['release_date'].dt.month

In [ ]:
tmdb_movies_df.columns

## Cleaning movie_budgets df(The Numbers Dataset)

In [ ]:
tn_movie_budgets_df.info()

In [ ]:
# checking for unique values in each column
for column in tn_movie_budgets_df.columns:
    print(f"\n{column} unique values:")
    print(tn_movie_budgets_df[column].unique())

In [ ]:
# converting currency strings to numeric 
print("\ntn_movie_budgets_df...")
for col in ['production_budget', 'domestic_gross', 'worldwide_gross']:
        if col in tn_movie_budgets_df.columns and tn_movie_budgets_df[col].dtype == 'object':
            tn_movie_budgets_df[col] = tn_movie_budgets_df[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
            tn_movie_budgets_df[col] = pd.to_numeric(tn_movie_budgets_df[col], errors='coerce')

In [ ]:
# checking 
tn_movie_budgets_df.head(10)

In [ ]:
# converting release date to datetime format
tn_movie_budgets_df['release_date']= pd.to_datetime(tn_movie_budgets_df['release_date'])

In [ ]:
# Extract release year and month from tn budget
tn_movie_budgets_df['release_year'] = tn_movie_budgets_df['release_date'].dt.year
tn_movie_budgets_df['release_month'] = tn_movie_budgets_df['release_date'].dt.month


In [ ]:
tn_movie_budgets_df.columns

## Feature engineering

### Calculating review counts from rt_reviews_clean_df

In [ ]:
# computing review counts and freshness score
review_counts= rt_reviews_clean_df.groupby('id').agg(
    Review_counts= ('fresh', 'count'),
    Freshness_score= ('fresh', lambda x: (x== 'fresh').sum()/ len(x))).reset_index()
review_counts

### Merging review_counts and rt_movie_info_clean df

In [ ]:
rt_combined_df= pd.merge(rt_movie_info_clean, review_counts, on= 'id', how= 'left')
rt_combined_df

### Merging tmdb_movies_df and tn_movie_budgets_df


In [ ]:
# renaming tmdb title column to tmdb_title for differentiation
tmdb_movies_df.rename(columns= {'title': 'tmdb_title'}, inplace= True)

In [ ]:
# renaming movie column in tn_movie_budgets_df for easy understanding when merging with tmdb
tn_movie_budgets_df.rename(columns={ 'movie': 'title'}, inplace= True)

In [ ]:
# merging tn_movie_budgets_df and tmdb_movies_df on title and release year
tmdb_tn_merged= pd.merge(tn_movie_budgets_df, tmdb_movies_df, left_on= ['title', 'release_year'], right_on=['tmdb_title', 'release_year'], how= 'left')
tmdb_tn_merged

In [ ]:
# Rename id_x for merging
tmdb_tn_merged.rename(columns={'id_x': 'id'}, inplace=True)

In [ ]:
merged_2= pd.merge(tmdb_tn_merged, rt_combined_df[['id', 'genre', 'rating', 'Review_counts', 'Freshness_score']], on='id', how='left')
merged_2

### Final feature engineering

In [ ]:
merged_2['Profit']= merged_2['worldwide_gross']- merged_2['production_budget']
merged_2['ROI'] = merged_2['Profit'] / merged_2['production_budget']
merged_2['Domestic_vs_Worldwide_Ratio'] = merged_2['domestic_gross'] / merged_2['worldwide_gross']
merged_2['Release_Month'] = merged_2['release_date_x'].dt.month
merged_2['Day_of_Week'] = merged_2['release_date_x'].dt.day_name()


In [ ]:
merged_2.head(10)

In [ ]:
merged_2.info()

In [ ]:
# Drop suffix columns and low-value metadata
redundant_cols = [col for col in merged_2.columns if col.endswith('_x') or col.endswith('_y')]
low_value_cols = [
    'id_y', 'tmdb_title', 'original_title', 'original_language', 'currency',
    'dvd_date', 'release_date_y', 'release_date_x', 'rating', 'writer',
    'synopsis', 'theater_date', 'popularity', 'vote_count', 'vote_average','Unnamed: 0','genre_ids'
]
drop_columns = list(set(redundant_cols + low_value_cols))
merged_2.drop(columns=drop_columns, errors='ignore', inplace=True)
merged_2.tail()

### Director and Writer Metrics

In [ ]:
review_metrics = rt_reviews_clean_df.groupby('id').agg(
    review_count=('fresh', 'count'),
    freshness_score=('fresh', lambda x: (x == 'fresh').sum() / len(x))
).reset_index()

In [ ]:
# Merge review metrics with director and writer
review_metrics_full = pd.merge(rt_combined_df[['id', 'director', 'writer']], review_metrics, on='id', how='left')
review_metrics_full

In [ ]:
# Group by director
director_metrics = review_metrics_full.groupby('director').agg(
    director_avg_freshness=('freshness_score', 'mean'),
    director_avg_reviews=('review_count', 'mean'),
    director_movie_count=('id', 'count')
).reset_index()

# Group by writer
writer_metrics = review_metrics_full.groupby('writer').agg(
    writer_avg_freshness=('freshness_score', 'mean'),
    writer_avg_reviews=('review_count', 'mean'),
    writer_movie_count=('id', 'count')
).reset_index()

In [ ]:
director_metrics

In [ ]:
writer_metrics

In [ ]:
# top directors avg reviews
top_directors_by_reviews = director_metrics.sort_values(by='director_avg_reviews', ascending=False).head(10)
top_directors_by_reviews

In [ ]:
# top writers avg reviews
top_writers_by_reviews = writer_metrics.sort_values(by='writer_avg_reviews', ascending=False).head(10)
top_writers_by_reviews
